In [ ]:
!python.exe -m pip install --upgrade pip

In [ ]:
# Make sure paths are correct for the imports

import os
import sys

notebook_dir = os.path.abspath("")
parent_dir = os.path.dirname(notebook_dir)
grandparent_dir = os.path.dirname(parent_dir)


sys.path.append(grandparent_dir)

In [ ]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    AzureChatPromptExecutionSettings,  # noqa: F401
    AzureTextCompletion,
    OpenAIChatCompletion,
    OpenAIChatPromptExecutionSettings,  # noqa: F401
    OpenAITextCompletion,
    OpenAITextPromptExecutionSettings,  # noqa: F401
)
from semantic_kernel.contents import ChatHistory  # noqa: F401


In [ ]:
from bing_search_agent import BingSearchAgent

bing_search_agent = BingSearchAgent()

azure_pattern_opt_triage_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="bing_search_agent", 
    instructions=(
        "You are an expert in Azure data and AI architecture and optimization. "
        "Your task is to evaluate user requests for specific Azure data or AI patterns and forward them to the appropriate specialized agents for targeted assistance. "
    ),
    plugins=[
        core_comps_agent,
        cost_opt_agent,
        perf_opt_agent,
        bing_search_agent,  # Add Bing agent as a plugin
    ],
)


In [ ]:
kernel = Kernel()

In [ ]:
from services import Service

from service_settings import ServiceSettings

service_settings = ServiceSettings()

# Select a service to use for this notebook (available services: OpenAI, AzureOpenAI, HuggingFace)
selectedService = (
    Service.AzureOpenAI
    if service_settings.global_llm_service is None
    else Service(service_settings.global_llm_service.lower())
)
print(f"Using service type: {selectedService}")



Using service type: Service.AzureOpenAI


In [ ]:
# Remove all services so that this cell can be re-run without restarting the kernel
kernel.remove_all_services()

service_id = None
if selectedService == Service.OpenAI:
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service_id = "default"
    kernel.add_service(
        OpenAIChatCompletion(
            service_id=service_id,
        ),
    )
    
elif selectedService == Service.AzureOpenAI:
    from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

    service_id = "default"
    kernel.add_service(
        AzureChatCompletion(
            service_id=service_id,
        ),
    )

In [ ]:
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, OpenAIChatCompletion
from semantic_kernel.contents import ChatHistory, ChatMessageContent, ImageContent, TextContent


In [ ]:
core_comps_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="CoreComps", # Renamed from CoreComponentsAgent
    instructions="Identify and explain the core Azure services that form the backbone of a requested data or AI pattern. Explain their individual roles and how they integrate."
)

cost_opt_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="CostOpt", # Renamed from CostOptimizationAgent
    instructions="Outline methods to minimize Azure consumption costs for a given data or AI pattern. This includes considerations like resource sizing, pricing tiers, reservations, auto-scaling, and data lifecycle management."
)

perf_opt_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="PerfOpt", # Renamed from PerformanceOptimizationAgent
    instructions="Detail techniques to maximize the speed and efficiency of a data or AI pattern. Address aspects such as data ingestion, processing, model inference, query performance, and caching."
)


azure_pattern_opt_triage_agent = ChatCompletionAgent(
    service=AzureChatCompletion(),
    name="agent", # Renamed from AzurePatternOptimizerTriageAgent
    instructions=(
        "You are an expert in Azure data and AI architecture and optimization. "
        "Your task is to evaluate user requests for specific Azure data or AI patterns and forward them to the appropriate specialized agents for targeted assistance. "
        "After gathering information from the specialized agents, provide the full, comprehensive answer to the user containing all relevant information."
    ),
    plugins=[
        core_comps_agent,
        cost_opt_agent,
        perf_opt_agent,
        bing_search_agent
    ],
)

In [ ]:
thread: ChatHistoryAgentThread = None


In [ ]:

user_input = "what is the best azure pattern for data in databricks with 300 production on prem sql databases that are overloaded and moving that into the cloud" 

In [ ]:
response = await azure_pattern_opt_triage_agent.get_response(
            messages=user_input,
            thread=thread,
        )

In [ ]:
print(response.message.content)


Below is a consolidated, end-to-end “Lakehouse” pattern on Azure that lets you migrate and modernize 300 on-prem SQL Server databases into a single, highly scalable, efficient, and cost-controlled Databricks-based platform. It covers:

1. Core Azure components  
2. Cost-optimization levers  
3. Performance-optimization techniques  

— — —  
1. Core Azure Components & Roles  

A. Ingestion & Replication  
 • Azure Data Factory (ADF) + Self-Hosted Integration Runtime  
    – Securely connect to all 300 on-prem SQL Servers.  
    – Perform initial full loads (Copy Activity or DMS) and incremental CDC loads.  

 • Azure Database Migration Service (optional for initial lift-and-shift)  
    – Migrate schemas and full data into Azure SQL Managed Instance/DB, then switch to ADF CDC.  

B. Landing Zone (Raw / Bronze)  
 • Azure Data Lake Storage Gen2  
    – Store raw extracts as Parquet or Delta files, partitioned by source DB and ingestion date.  
    – Use ACLs, encryption, and lifecycle po